In [1]:
import pennylane as qml
from pennylane import numpy as np

In [15]:
np.random.seed(499122177)

In [2]:
def init_graph():
    n, e = [int(x) for x in input("Enter the number of vertices and edges: ").split()]
    edges = []
    for _ in range(e):
        u, v = [int(x) for x in input().split()]
        edges.extend([u, v])

    return [n, e]

In [4]:
def U_B(qubits, beta):
    for qubit in range(qubits):
        qml.RX(2 * beta, wires = qubit)

In [5]:
def U_C(edges, gamma):
    for edge in edges:
        qml.CNOT(wires = edge)
        qml.RZ(gamma, wires = edge[1])
        qml.CNOT(wires = edge)

In [6]:
def bitstring_to_int(bit_string):
    return int(2 ** np.arange(len(bit_string)) @ bit_string[::-1])

In [7]:
qubits, edges = init_graph()

Enter the number of vertices and edges:  4 4
 0 1
 0 3
 1 2
 2 3


In [8]:
dev = qml.device('default.qubit', wires = qubits)

In [13]:
@qml.set_shots(1024)
@qml.qnode(dev)
def ckt(qubits, edges, gammas, betas, return_samples = False):
    assert(len(gammas) == qubits)
    assert(len(betas) == qubits)

    for i in range(qubits):
        qml.Hadamard(wires = i)
    for gamma, beta in zip(gammas, betas):
        U_C(qubits, gamma)
        U_B(qubits, beta)

    if return_samples:
        qml.sample()

    H = qml.sum(*(qml.Z(w1) @ qml.Z(w2) for w1, w2 in edges))
    return qml.expval(H)

In [14]:
def objective(edges, params):
    return -0.5 * (len(edges) - ckt(*params))

In [16]:
def qaoa_maxcut(n_layers = 1):

    init_params = 0.01 * np.random.rand(2, n_layers, requires_grad = True)
    opt = qml.AdagradOptimizer(stepsize = 0.5)

    params = init_params.copy()
    steps = 30

    for i in range(steps):
        params = opt.step(objective, params)
        if (i + 1) % 5 == 0:
            print(f"Objective after step {i + 1:3d}: {-objective(params): .7f}")

    
            